First rounds of pareto analysis and incorporation to WC batch sims have been done and showed limited change to validations like cell health and proteomics, while positively meeting homeostatic need at all timepoints and improving the kinetics term. However, one huge caveat in this analysis was the awful fit of central metabolism flux to validation data Toya 2010. Though Pearson correlation remain quite high, the coefficieny of determination R-squared was so bad that all batch sims had negative COD R-squared. This is a huge problem because it may suggest that the pareto optimized weight combinations are over optimizing for some parts of the model at the cost of central metabolism and that meeting the homeostatic need does not correlate to having a good central metabolism fit. In this notebook, I want to investigate the relationship amongst the objective weights in respect to the central metabolism fit and homeostatic objective term

In [4]:
import pandas as pd
import os, dill
import numpy as np
from typing import Iterable, Optional, Mapping, cast
import altair as alt
import polars as pl
import pandas as pd

from ecoli.analysis.single.centralCarbonMetabolismScatter import FLUX_UNITS
from ecoli.processes.metabolism_redux_classic import NetworkFlowModel, FlowResult
import pickle
from fsspec import open as fsspec_open
from wholecell.utils import units, toya

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

alt.renderers.enable("browser")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


RendererRegistry.enable('browser')

A pareto_results.csv file lists all the sampled weight combinations and results of running standalone FBA using them. Though the standalone FBA's central metabolism fit doesn't reflect that in a WC sim. I still want to see if there is any correlation between sampled weights and the central metabolism fit to see if there is any pattern that can be observed.

In [36]:
csv_path = f'notebooks/Heena notebooks/Metabolism_New Genes/pareto_results_init_2000samples/pareto_results.csv'
pareto_results = pd.read_csv(csv_path)

In [38]:
cols = ['lambda_sec','lambda_eff','lambda_kin','lambda_div','toya_r_squared', 'obj_homeo']
lambda_cols = ['lambda_sec','lambda_eff','lambda_kin','lambda_div']

df_plot = pareto_results[cols].copy()
df_plot.loc[:, lambda_cols] = df_plot.loc[:, lambda_cols].apply(lambda x: np.log10(x))

mat = df_plot.corr()

long = mat.stack().reset_index()
long.columns = ["col_a", "col_b", "r"]
long["r"] = long["r"].round(4)

base = alt.Chart(long).encode(
    x=alt.X("col_a:N", title=None, sort=cols, axis=alt.Axis(labelAngle=-30)),
    y=alt.Y("col_b:N", title=None, sort=cols),
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1]),
        legend=alt.Legend(title="Pearson r"),
    ),
    tooltip=["col_a", "col_b", "r"],
)

text = base.mark_text(fontSize=12).encode(
    text=alt.Text("r:Q", format=".2f"),
    color=alt.condition(
        "abs(datum.r) > 0.6", alt.value("white"), alt.value("black")
    ),
)

chart = (heatmap + text).properties(
    title="Correlation Matrix of Log10(Weight Combos) to FBA Toya R² Init", width=420, height=420
)

chart.save('notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/'
           'central_metabolism/init_correlation_matrix_weights_toya_r2.svg')

chart

alt.LayerChart(...)

### Partial and semipartial correlation (log₁₀ weights vs outcomes)

The heatmap above is **pairwise Pearson r** on the full table. Each λ is **not** independent of the others in how they affect the LP solution, so a **partial** correlation measures the linear association between one log₁₀(λ) and an outcome **after removing linear dependence on the other three** log₁₀(λ).

- **Partial r** (predictor *i*, outcome): correlate residuals of (*i* ~ other λs) with residuals of (outcome ~ other λs). Symmetric in predictor/outcome for linear partial correlation.
- **Semipartial r** (predictor *i*, outcome): correlate the outcome *Y* with the residual of (λᵢ ~ other λs). This matches the usual “unique linear association of λᵢ with *Y* given the other weights” (incremental R² is related to the square of this in OLS).

Both use **linear** controls only; they complement the raw correlation matrix but do not capture nonlinear interactions.

In [39]:
from IPython.display import display
from scipy.stats import pearsonr
import statsmodels.api as sm

# Same log10(λ) frame as the correlation matrix above
_pc_cols = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div", "toya_r_squared", "obj_homeo"]
_pc_lambda_cols = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
_df_pc = pareto_results[_pc_cols].copy()
_df_pc.loc[:, _pc_lambda_cols] = _df_pc.loc[:, _pc_lambda_cols].apply(np.log10)

_outcomes = ["toya_r_squared", "obj_homeo"]


def _partial_pearson_with_p(
    x: np.ndarray, y: np.ndarray, Z: pd.DataFrame
) -> tuple[float, float]:
    """Partial Pearson r between x and y, controlling linearly for columns of Z."""
    Zm = sm.add_constant(Z.to_numpy(), has_constant="add")
    res_x = sm.OLS(x, Zm).fit().resid
    res_y = sm.OLS(y, Zm).fit().resid
    r, p = pearsonr(res_x, res_y)
    return float(r), float(p)


def _semipartial_pearson_with_p(
    x: np.ndarray, y: np.ndarray, Z: pd.DataFrame
) -> tuple[float, float]:
    """Semipartial r: corr(y, residual of x | Z). Unique linear overlap of x with y given Z."""
    Zm = sm.add_constant(Z.to_numpy(), has_constant="add")
    res_x = sm.OLS(x, Zm).fit().resid
    r, p = pearsonr(y, res_x)
    return float(r), float(p)


_rows = []
for outcome in _outcomes:
    y = _df_pc[outcome].to_numpy(dtype=float)
    for lam in _pc_lambda_cols:
        others = [c for c in _pc_lambda_cols if c != lam]
        Z = _df_pc[others]
        x = _df_pc[lam].to_numpy(dtype=float)
        pr, pp = _partial_pearson_with_p(x, y, Z)
        sr, sp = _semipartial_pearson_with_p(x, y, Z)
        _rows.append(
            {
                "predictor": lam,
                "outcome": outcome,
                "partial_r": pr,
                "partial_p": pp,
                "semipartial_r": sr,
                "semipartial_p": sp,
            }
        )

partial_corr_df = pd.DataFrame(_rows)
display(partial_corr_df)

# Long format for Altair: two facets (partial vs semipartial)
_long = partial_corr_df.melt(
    id_vars=["predictor", "outcome"],
    value_vars=["partial_r", "semipartial_r"],
    var_name="kind",
    value_name="r",
)
_long["kind"] = _long["kind"].map(
    {"partial_r": "partial (residual vs residual)", "semipartial_r": "semipartial (Y vs λ residual)"}
)

_base_pc = alt.Chart(_long).encode(
    x=alt.X("outcome:N", title=None, sort=_outcomes),
    y=alt.Y("predictor:N", title=None, sort=_pc_lambda_cols),
)

_heatmap_pc = _base_pc.mark_rect().encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1]),
        legend=alt.Legend(title="Pearson r"),
    ),
    tooltip=["predictor", "outcome", "kind", alt.Tooltip("r:Q", format=".4f")],
)

_text_pc = _base_pc.mark_text(fontSize=11).encode(
    text=alt.Text("r:Q", format=".2f"),
    color=alt.condition(
        "abs(datum.r) > 0.55", alt.value("white"), alt.value("black")
    ),
)

_chart_pc = (
    (_heatmap_pc + _text_pc)
    .properties(width=220, height=280)
    .facet(
        column=alt.Column(
            "kind:N",
            title=None,
            sort=[
                "partial (residual vs residual)",
                "semipartial (Y vs λ residual)",
            ],
        )
    )
    .properties(
        title="Partial and semipartial Pearson r: log10(λ) vs outcomes (control other 3 λs) Init"
    )
)

_out_partial = (
    "notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/"
    "central_metabolism/init_partial_semipartial_corr_weights_outcomes.svg"
)
_chart_pc.save(_out_partial)
print(f"Saved: {_out_partial}")
_chart_pc

,predictor,outcome,partial_r,partial_p,semipartial_r,semipartial_p
0,lambda_sec,toya_r_squared,0.451362,5.711004e-101,0.275542,3.556211e-36
1,lambda_eff,toya_r_squared,0.374641,1.150865e-67,0.220115,2.278509e-23
2,lambda_kin,toya_r_squared,-0.810379,0.000000e+00,-0.753451,0.000000e+00
3,lambda_div,toya_r_squared,-0.000792,9.717605e-01,-0.000431,9.846144e-01
4,lambda_sec,obj_homeo,0.650471,5.860865e-241,0.584347,1.767791e-183
5,lambda_eff,obj_homeo,0.330549,3.392047e-52,0.238973,2.275707e-27
6,lambda_kin,obj_homeo,0.495081,3.669472e-124,0.388796,3.602453e-73
7,lambda_div,obj_homeo,0.009780,6.620161e-01,0.006674,7.654941e-01


Saved: notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/central_metabolism/init_partial_semipartial_corr_weights_outcomes.svg


alt.FacetChart(...)

### 1. Pairwise log-ratio correlations

Individual weight correlations capture marginal effects, but the optimiser sees **relative** weights — how much one term is penalised compared to another. Each ratio λᵢ/λⱼ is the log-difference `log10(λᵢ) − log10(λⱼ)`.
Here we compute all 6 pairwise log-ratios and their Pearson r with `toya_r_squared` and `obj_homeo`.

In [40]:
from itertools import combinations

_ratio_lambda_cols = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
_ratio_outcomes    = ["toya_r_squared", "obj_homeo"]

# Build log10 frame
_df_ratio = pareto_results[_ratio_lambda_cols + _ratio_outcomes].copy()
_df_ratio[_ratio_lambda_cols] = _df_ratio[_ratio_lambda_cols].apply(np.log10)

# Compute all pairwise log-ratios (log10(λi) - log10(λj))
_ratio_rows = []
for lam_i, lam_j in combinations(_ratio_lambda_cols, 2):
    ratio_name = f"{lam_i} / {lam_j}"
    ratio_vals = _df_ratio[lam_i] - _df_ratio[lam_j]  # log-difference == log of ratio
    for outcome in _ratio_outcomes:
        r, p = pearsonr(ratio_vals, _df_ratio[outcome])
        _ratio_rows.append({
            "ratio": ratio_name,
            "outcome": outcome,
            "r": float(r),
            "p": float(p),
        })

ratio_corr_df = pd.DataFrame(_ratio_rows)
display(ratio_corr_df.pivot(index="ratio", columns="outcome", values="r").round(3))

# ── Altair heatmap ────────────────────────────────────────────────────────────
_ratio_order = [f"{i} / {j}" for i, j in combinations(_ratio_lambda_cols, 2)]

_base_ratio = alt.Chart(ratio_corr_df).encode(
    x=alt.X("outcome:N", title=None, sort=_ratio_outcomes),
    y=alt.Y("ratio:N",   title=None, sort=_ratio_order),
)

_heatmap_ratio = _base_ratio.mark_rect().encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1]),
        legend=alt.Legend(title="Pearson r"),
    ),
    tooltip=["ratio", "outcome", alt.Tooltip("r:Q", format=".4f"), alt.Tooltip("p:Q", format=".2e")],
)

_text_ratio = _base_ratio.mark_text(fontSize=11).encode(
    text=alt.Text("r:Q", format=".2f"),
    color=alt.condition(
        "abs(datum.r) > 0.55", alt.value("white"), alt.value("black")
    ),
)

_chart_ratio = (_heatmap_ratio + _text_ratio).properties(
    width=200, height=280,
    title="Pearson r: log10(λᵢ/λⱼ) vs outcomes Init"
)

_out_ratio = (
    "notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/"
    "central_metabolism/init_log_ratio_corr_weights_outcomes.svg"
)
_chart_ratio.save(_out_ratio)
print(f"Saved: {_out_ratio}")
_chart_ratio


outcome,obj_homeo,toya_r_squared
ratio,,
lambda_eff / lambda_div,0.176,0.151
lambda_eff / lambda_kin,-0.086,0.685
lambda_kin / lambda_div,0.263,-0.529
lambda_sec / lambda_div,0.341,0.190
lambda_sec / lambda_eff,0.145,0.021
lambda_sec / lambda_kin,0.050,0.775


Saved: notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/central_metabolism/init_log_ratio_corr_weights_outcomes.svg


alt.LayerChart(...)

### 4. Conditional correlation by homeostatic cluster

Global correlations average over all points, which can mask local conditional relationships. Here we split the Pareto set into **low** and **high** `obj_homeo` clusters (median split) and rerun the raw Pearson r heatmap within each. If `lambda_sec` only becomes influential within one cluster, that will show up here.

In [45]:
_clust_cols    = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div", "toya_r_squared", "obj_homeo", "obj_kin"]
_clust_lambdas = ["lambda_sec", "lambda_eff", "lambda_kin", "lambda_div"]
_clust_outcomes = ["toya_r_squared", "obj_homeo"]

# Build log10 frame and add cluster label (median split on obj_kin)
_df_clust = pareto_results[_clust_cols].copy()
_df_clust[_clust_lambdas] = _df_clust[_clust_lambdas].apply(np.log10)
_df_clust["cluster"] = np.where(
    (22.5 <= _df_clust["obj_kin"]) & (_df_clust["obj_kin"]<= 30), "cluster region", "outside"
)

print(_df_clust["cluster"].value_counts())

# Compute Pearson r for each cluster
_clust_rows = []
for cluster_name, group in _df_clust.groupby("cluster"):
    for lam in _clust_lambdas:
        for outcome in _clust_outcomes:
            r, p = pearsonr(group[lam], group[outcome])
            _clust_rows.append({
                "predictor": lam,
                "outcome": outcome,
                "cluster": cluster_name,
                "r": float(r),
                "p": float(p),
            })

clust_corr_df = pd.DataFrame(_clust_rows)

# Print pivot for quick inspection
for _c in ["cluster region", "outside"]:
    print(f"\n--- {_c} ---")
    display(
        clust_corr_df[clust_corr_df["cluster"] == _c]
        .pivot(index="predictor", columns="outcome", values="r")
        .round(3)
    )

# ── Altair: facet by cluster, heatmap per facet ───────────────────────────────
_base_clust = alt.Chart(clust_corr_df).encode(
    x=alt.X("outcome:N",   title=None, sort=_clust_outcomes),
    y=alt.Y("predictor:N", title=None, sort=_clust_lambdas),
)

_heatmap_clust = _base_clust.mark_rect().encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1]),
        legend=alt.Legend(title="Pearson r"),
    ),
    tooltip=["predictor", "outcome", "cluster",
             alt.Tooltip("r:Q", format=".4f"), alt.Tooltip("p:Q", format=".2e")],
)

_text_clust = _base_clust.mark_text(fontSize=11).encode(
    text=alt.Text("r:Q", format=".2f"),
    color=alt.condition(
        "abs(datum.r) > 0.55", alt.value("white"), alt.value("black")
    ),
)

_chart_clust = (
    (_heatmap_clust + _text_clust)
    .properties(width=200, height=220)
    .facet(
        column=alt.Column(
            "cluster:N", title=None, sort=["cluster region", "outside"]
        )
    )
    .properties(
        title="Pearson r by obj_homeo cluster (median split): log10(λ) vs outcomes Init"
    )
)

_out_clust = (
    "notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/"
    "central_metabolism/conditional_corr_by_kinetic_region_init.svg"
)
_chart_clust.save(_out_clust)
print(f"Saved: {_out_clust}")
_chart_clust


cluster
outside           1373
cluster region     627
Name: count, dtype: int64

--- cluster region ---


outcome,obj_homeo,toya_r_squared
predictor,,
lambda_div,0.023,-0.070
lambda_eff,0.098,0.301
lambda_kin,0.459,0.478
lambda_sec,0.734,-0.345



--- outside ---


outcome,obj_homeo,toya_r_squared
predictor,,
lambda_div,-0.012,0.003
lambda_eff,0.252,0.503
lambda_kin,0.339,-0.746
lambda_sec,0.653,0.202


Saved: notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/central_metabolism/conditional_corr_by_kinetic_region_init.svg


alt.FacetChart(...)